In [ ]:
import os
import numpy as np
from skimage.io import imread, imsave
from skimage.color import rgb2gray
from skimage.feature import hog
from skimage.transform import resize
from skimage import img_as_ubyte
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split, learning_curve 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier                   
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay


# ----------------------------
# Load dataset
data_dir = './dataset'
images = []
labels = []

files = os.listdir(data_dir)
for file in files:
    if file.endswith('.jpg'):
        img_path = os.path.join(data_dir, file)

        # Labels: 0=cat, 1=dog
        if file.startswith('cat'):
            labels.append(0)
        elif file.startswith('dog'):
            labels.append(1)
        else:
            continue

        # Read + preprocess
        img = imread(img_path)
        if img.ndim == 3:
            img = rgb2gray(img)
        img_resized = resize(img, (128, 128), anti_aliasing=True)
        images.append(img_resized)

images = np.array(images)
labels = np.array(labels)

print('Images shape:', images.shape)
print('Labels shape:', labels.shape)
print(f'Number of cats: {np.sum(labels == 0)}')
print(f'Number of dogs: {np.sum(labels == 1)}')


# ----------------------------
# Extract HOG features
def extract_hog_features(images, visualize=False,
                         pixels_per_cell=(8, 8),
                         cells_per_block=(2, 2),
                         orientations=9):
    hog_features = []
    hog_images = []
    for img in images:
        if visualize:
            features, hog_image = hog(img,
                                      orientations=orientations,
                                      pixels_per_cell=pixels_per_cell,
                                      cells_per_block=cells_per_block,
                                      visualize=True,
                                      block_norm='L2-Hys')
            hog_features.append(features)
            hog_images.append(hog_image)
        else:
            features = hog(img,
                           orientations=orientations,
                           pixels_per_cell=pixels_per_cell,
                           cells_per_block=cells_per_block,
                           visualize=False,
                           block_norm='L2-Hys')
            hog_features.append(features)

    hog_features = np.array(hog_features)
    hog_images = np.array(hog_images) if visualize else None
    return hog_features, hog_images


# ----------------------------
# Split dataset
def split_dataset(images, labels, test_size=0.2):
    idx_train, idx_test = train_test_split(
        np.arange(len(images)),
        test_size=test_size,
        stratify=labels,
        random_state=42
    )
    y_train = labels[idx_train]
    y_test = labels[idx_test]
    return idx_train, idx_test, y_train, y_test



In [ ]:

# ------------------------------------------------------------------------
#############################  TRAIN MODELS  #############################
# ------------------------------------------------------------------------
# Decision Tree
# X_train: training features (HOG features)
# y_train: training labels (0=cat, 1=dog)
# max_depth: maximum depth of the tree
# min_samples_split: minimum samples required to split a node
# min_samples_leaf: minimum samples required to be at a leaf node
def train_decision_tree(X_train, y_train, max_depth=20, min_samples_split=5, min_samples_leaf=2):
    clf = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split,
                                 min_samples_leaf=min_samples_leaf, random_state=42)
    clf.fit(X_train, y_train)  # Train the decision tree on input data
    return clf  # Return the trained model

# Random Forest
# X_train: training features (HOG features)
# y_train: training labels (0=cat, 1=dog)
# n_estimators: number of decision trees in the forest
# max_depth: maximum depth of each tree
def train_random_forest(X_train, y_train, n_estimators=100, max_depth=20):
    clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    clf.fit(X_train, y_train)  # Train the random forest on the training data
    return clf  # Return the trained model

# Linear Regression
# X_train: training features
# y_train: training labels
def train_linear_regression(X_train, y_train):
    clf = LinearRegression()
    clf.fit(X_train, y_train)  # Train the linear regression model
    return clf  # Return the trained model

# Thresholded Linear Regression
# Converts continuous regression output to binary class (0 or 1)
# threshold: threshold for classification
class ThresholdedLinearRegression(BaseEstimator, ClassifierMixin):
    def __init__(self, threshold=0.5):
        self.threshold = threshold  # Classification threshold
        self.model = LinearRegression()  # Internal linear regression model

    # Fit the model on training data
    # X: training features
    # y: training labels
    def fit(self, X, y):
        self.model.fit(X, y)
        return self

    # Predict binary classes
    # X: input features for prediction
    # Returns an array of 0s and 1s
    def predict(self, X):
        y_cont = self.model.predict(X)  # Continuous prediction
        return (y_cont >= self.threshold).astype(int)  # Apply threshold to get binary classes

# Helper function to convert Linear Regression outputs to binary
# model: trained linear regression model
# X: input features
# threshold: threshold for converting continuous output to class
# Returns: y_pred (binary classes), y_cont (continuous output)
def lr_predict_to_binary(model, X, threshold=0.5):
    y_cont = model.predict(X)  # Continuous prediction
    return (y_cont >= threshold).astype(int), y_cont  # Convert to binary and return both



In [ ]:

# ------------------------------------------------------------------------
#############################  EVALUTW  #############################
# ------------------------------------------------------------------------
# Evaluate model accuracy
# clf: trained classifier (DecisionTree, RandomForest, or Linear Regression)
# X_test: test features
# y_test: test labels
# Returns: accuracy (float), y_pred (predicted labels)
def evaluate_model(clf, X_test, y_test):
    y_pred = clf.predict(X_test)  # Predict labels for test set
    accuracy = accuracy_score(y_test, y_pred)  # Compute accuracy
    return accuracy, y_pred  # Return accuracy and predicted labels

# Plot confusion matrix
# y_true: true labels
# y_pred: predicted labels
# title: title for the plot
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)  # Compute confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Cat', 'Dog'])  # Set class labels
    disp.plot(cmap=plt.cm.Blues, values_format='d')  # Display the matrix
    plt.title(title)
    plt.show()

# Plot learning curve
# estimator: trained or untrained model to evaluate (DecisionTree, RandomForest, etc.)
# X: training features
# y: training labels
# title: title for the plot
# cv: number of cross-validation folds
# train_sizes: sizes of training sets to use
def plot_learning_curve(estimator, X, y, title, cv=5, train_sizes=np.linspace(0.1, 1.0, 5)):
    # Compute learning curve
    train_sizes, train_scores, test_scores = learning_curve(estimator, X, y, cv=cv,
                                                            train_sizes=train_sizes,
                                                            scoring='accuracy')
    # Compute mean accuracy for each training set size
    train_mean = np.mean(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)

    # Plot learning curve
    plt.figure()
    plt.plot(train_sizes, train_mean, 'o-', label='Training Accuracy')
    plt.plot(train_sizes, test_mean, 'o-', label='Cross-Validation Accuracy')
    plt.title(f'Learning Curve: {title}')
    plt.xlabel('Number of Training Examples')
    plt.ylabel('Accuracy')
    plt.legend(loc='best')
    plt.grid(True)
    plt.show()



In [ ]:
# Extract HOG features from images for training and visualization
hog_features, hog_images = extract_hog_features(images, visualize=True)

# Split dataset into training and testing sets
idx_train, idx_test, y_train, y_test = split_dataset(images, labels, test_size=0.2)
X_train, X_test = hog_features[idx_train], hog_features[idx_test]

# Train Decision Tree, Random Forest, and Linear Regression models
dt = train_decision_tree(X_train, y_train)
rf = train_random_forest(X_train, y_train)         
lr = train_linear_regression(X_train, y_train)     

# Evaluate models on test set and get predictions
dt_acc, dt_pred = evaluate_model(dt, X_test, y_test)
rf_acc, rf_pred = evaluate_model(rf, X_test, y_test)               
lr_pred, lr_cont = lr_predict_to_binary(lr, X_test, threshold=0.5) 
lr_acc = accuracy_score(y_test, lr_pred)                           

# Print the accuracy of each model
print(f"Decision Tree Accuracy: {dt_acc:.4f}")
print(f"Random Forest Accuracy: {rf_acc:.4f}")     
print(f"Linear Regression Accuracy: {lr_acc:.4f}") 

# Plot confusion matrices to analyze model performance
plot_confusion(y_test, dt_pred, "Decision Tree Confusion Matrix")  
plot_confusion(y_test, rf_pred, "Random Forest Confusion Matrix")  
plot_confusion(y_test, lr_pred, "Linear Regression Confusion Matrix")  

# Plot learning curves to visualize training behavior
plot_learning_curve(DecisionTreeClassifier(max_depth=20, random_state=42),
                    X_train, y_train, "Decision Tree")
plot_learning_curve(RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42),
                    X_train, y_train, "Random Forest")
plot_learning_curve(ThresholdedLinearRegression(threshold=0.5),
                    X_train, y_train, "Linear Regression (thresholded)")


# Visualize sample images, their HOG features, and model predictions
classes = ['cat', 'dog']
sample_indices = idx_test[:6]   # show 6 samples max
fig, axes = plt.subplots(len(sample_indices), 2, figsize=(10, 3 * len(sample_indices)))
for i, idx in enumerate(sample_indices):
    true_lbl = y_test[np.where(idx_test == idx)[0][0]]
    dt_p = dt_pred[np.where(idx_test == idx)[0][0]]
    rf_p = rf_pred[np.where(idx_test == idx)[0][0]]
    lr_p = lr_pred[np.where(idx_test == idx)[0][0]]

    # Display original image with true label
    ax = axes[i, 0]
    ax.imshow(images[idx], cmap='gray')
    ax.set_title(f"True: {classes[true_lbl]}")
    ax.axis('off')

    # Display HOG image with predictions from all models
    ax = axes[i, 1]
    ax.imshow(hog_images[idx], cmap='gray')
    ax.set_title(f"HOG | DT:{classes[dt_p]} | RF:{classes[rf_p]} | LR:{classes[lr_p]}")
    ax.axis('off')

plt.tight_layout()
plt.show()  # Show all visualizations
